In [ ]:
"""
MODIFIED ONE - CLONED FROM THE ORIGINAL CODE

class_label is still 9; Since we used the 10 class model

If you want to run, models separately search for 
"""

'\nMODIFIED ONE - CLONED FROM THE ORIGINAL CODE\n\nclass_label is still 9; Since we used the 10 class model\n\nIf you want to run, models separately search for \n'

## <span style="color:red">Wait!!! Don't run the script right away..</span>
If you want to run models separately search for #LOADPICKLESANDRUN (to save time)

In [ ]:
#--------------------------------------------------------------------------------------------<Import libraries>--------------||
import numpy as np
import pandas as pd
import h5py
import os
import warnings
import torch
from torch.distributions import MultivariateNormal
from jakteristics import compute_features

#SUPPRESS WARNINGS
warnings.filterwarnings("ignore")

In [ ]:
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 4080 SUPER


In [ ]:
fileName = '001'

global_asprs_classes = [1, 2, 6, 9, 26] 
num_global_classes = len(global_asprs_classes)

asprs_to_idx = {cls: i for i, cls in enumerate(global_asprs_classes)}

In [ ]:
Input_path = rf"F:\Aditya\Tiles\AHN3 Tiles"

data = pd.read_csv(Input_path, sep=r'\s+', engine='python')
data.columns = data.columns.str.replace('//', '', regex=False).str.strip().str.lower()
if 'classification' in data.columns:
    data = data.rename(columns={'classification': 'label'})
if any (col in data.columns for col in ['nx', 'ny', 'nz']):
    data = data.drop(columns=['nx', 'ny', 'nz'])
    
data.columns  

Index(['x', 'y', 'z', 'r', 'g', 'b', 'intensity', 'return_number',
       'number_of_returns', 'edgeofflightline', 'label', 'overlap_flag',
       'scan_angle', 'point_source_id', 'gps_time', 'near_infrared'],
      dtype='object')

In [ ]:
data['x'] -= data['x'].min()
data['y'] -= data['y'].min()

In [ ]:
data = data.astype(float)
grouped = data.groupby(data['label'])

#--------------------------------------------------------------------------------------------<Check distribution>------------||
averages = grouped.mean()
variances = grouped.var()
averages

,x,y,z,r,g,b,intensity,return_number,number_of_returns,edgeofflightline,...,omnivariance,eigenentropy,anisotropy,planarity,linearity,PCA1,PCA2,surface_variation,sphericity,verticality
label,,,,,,,,,,,,,,,,,,,,,
0.0,626.753882,687.666546,115.812243,66.306121,80.162595,72.351804,59.872464,1.928857,3.569998,0.000216,...,0.139520,0.815068,0.670732,0.323057,0.347675,0.519549,0.323287,0.157164,0.329268,0.376167
1.0,552.405909,715.454703,106.578089,99.262824,104.657977,98.553518,237.782205,1.558291,1.558352,0.000987,...,0.041391,0.703737,0.985880,0.845008,0.140871,0.535736,0.456797,0.007468,0.014120,0.009174
2.0,441.577534,604.056658,113.837453,116.274078,116.123472,116.996808,156.755234,1.053216,1.132468,0.001192,...,0.051271,0.694459,0.965967,0.712689,0.253278,0.569864,0.411074,0.019062,0.034033,0.159634
3.0,768.252545,808.889856,103.128393,70.476823,79.005106,74.537969,129.617373,1.288660,1.288698,0.001509,...,0.049153,0.695643,0.979565,0.777569,0.201996,0.556754,0.432330,0.010916,0.020435,0.009516
4.0,431.685566,990.210654,101.751575,126.600776,130.133696,128.447368,143.566092,1.166717,1.276719,0.000000,...,0.053791,0.707415,0.951937,0.735791,0.216146,0.552737,0.420836,0.026427,0.048063,0.092197


## <span style="color:red">Run from here if the Kernel is getting trashed..</span>

In [ ]:
"""
#--------------------------------------------------------------------------------------------<Import libraries>--------------||
import numpy as np
import pandas as pd
import os
import warnings

#SUPPRESS WARNINGS
warnings.filterwarnings("ignore")
import pickle
import gc

fileName = "features_L003"

# Load CSV
data = pd.read_csv("ProcessedData/features_L003.csv")
data = data.drop(['Unnamed: 0'], axis=1).dropna().reset_index()
data = data.astype(float)

# Group data
grouped = data.groupby(data['label'])

# Save and unload `averages`
averages = grouped.mean()
with open("tempDumps/averages.pkl", "wb") as f:
    pickle.dump(averages, f)
del averages
gc.collect()

# Save and unload `variances`
variances = grouped.var()
with open("tempDumps/variances.pkl", "wb") as f:
    pickle.dump(variances, f)
del variances
gc.collect()

# Save and unload `grouped`
# Convert to dict-of-dfs before pickling
grouped_dict = {k: v for k, v in grouped}
with open("tempDumps/grouped.pkl", "wb") as f:
    pickle.dump(grouped_dict, f)
del grouped
del grouped_dict
gc.collect()
"""

print("Done")

Done


In [ ]:
def compute_covariance_matrix(data, regularization=1e-5):
    if data.shape[0] < 2:
        return np.eye(data.shape[1]) * regularization
    
    cov_matrix = np.cov(data, rowvar=False)
    cov_matrix += regularization * np.eye(cov_matrix.shape[0])
    
    return cov_matrix

In [ ]:
def fit(x_train, y_train):
    y_train = y_train.ravel()
    m = y_train.shape[0] 
    input_feature = x_train.shape[1]
    
    unique_labels = np.unique(y_train)
    num_classes = len(unique_labels)
    
    mu = np.zeros((num_classes, input_feature))
    sigma = np.zeros((num_classes, input_feature, input_feature))
    phi = np.zeros(num_classes)

    for idx, label_val in enumerate(unique_labels):
        indices = (y_train == label_val)
        x_class = x_train[indices, :]
        
        if len(x_class) > 1:
            phi[idx] = float(np.sum(indices)) / m
            mu[idx] = np.mean(x_class, axis=0)
            sigma[idx] = compute_covariance_matrix(x_class)
        else:
            phi[idx] = float(np.sum(indices)) / m
            mu[idx] = x_class[0] if len(x_class) == 1 else np.zeros(input_feature)
            sigma[idx] = np.eye(input_feature) * 1e-5
    
    return phi, mu, sigma, unique_labels

In [ ]:
has_nan = data.isnull().values.any()
print(has_nan)

#data.drop('index', axis=1, inplace=True)
data['label'] = data['label'].astype(int)
data.columns

False


Index(['x', 'y', 'z', 'r', 'g', 'b', 'intensity', 'return_number',
       'number_of_returns', 'edgeofflightline', 'overlap_flag', 'scan_angle',
       'point_source_id', 'gps_time', 'near_infrared', 'eigenvalue_sum',
       'omnivariance', 'eigenentropy', 'anisotropy', 'planarity', 'linearity',
       'PCA1', 'PCA2', 'surface_variation', 'sphericity', 'verticality',
       'label'],
      dtype='object')

##### <span style="color:red">Normalization : Exlcuding z (temporary, the iidea is to consider x,y for training)</span>

In [ ]:
#--------------------------------------------------------------------------------------------<Nomralization>-----------------||
from sklearn.preprocessing import MinMaxScaler #scikit-learn
scaler = MinMaxScaler()

"""columns_to_scale = ['z', 'eigenvalue_sum', 'omnivariance', 'eigenentropy',
       'anisotropy', 'planarity', 'linearity', 'PCA1', 'PCA2',
       'surface_variation', 'sphericity', 'verticality', 'nx', 'ny', 'nz']""" #Toronto

columns_to_scale = ['eigenvalue_sum', 'omnivariance', 'eigenentropy',
       'anisotropy', 'planarity', 'linearity', 'PCA1', 'PCA2',
       'surface_variation', 'sphericity', 'verticality'] #Toronto, excluded z

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[columns_to_scale])
scaled_df = pd.DataFrame(scaled_data, columns=columns_to_scale)
data[columns_to_scale] = scaled_df
data

,x,y,z,r,g,b,intensity,return_number,number_of_returns,edgeofflightline,...,eigenentropy,anisotropy,planarity,linearity,PCA1,PCA2,surface_variation,sphericity,verticality,label
0,0.120000,0.140000,105.499001,53.0,71.0,83.0,230.0,1.0,1.0,0.0,...,0.430414,0.999381,0.501840,0.497701,0.496085,0.668535,0.001226,0.000619,0.000193,1
1,0.582000,0.157000,105.526001,47.0,65.0,77.0,122.0,1.0,1.0,0.0,...,0.517771,0.999591,0.444475,0.555255,0.536133,0.615592,0.000843,0.000409,0.000247,1
2,0.330000,0.378000,105.497002,63.0,80.0,88.0,680.0,1.0,1.0,0.0,...,0.521136,0.999601,0.616214,0.383577,0.425491,0.762628,0.000735,0.000399,0.000157,1
3,0.068000,0.599000,105.510002,55.0,72.0,82.0,642.0,1.0,1.0,0.0,...,0.492219,0.999563,0.513108,0.486615,0.488866,0.678367,0.000858,0.000437,0.000196,1
4,1.075000,0.122000,105.522003,35.0,52.0,62.0,105.0,1.0,1.0,0.0,...,0.536236,0.999177,0.328931,0.670358,0.625967,0.495608,0.001841,0.000823,0.000517,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40895469,1020.776001,1270.646973,99.528000,71.0,79.0,68.0,28.0,1.0,1.0,0.0,...,0.670405,0.994140,0.888687,0.105815,0.286691,0.941413,0.009181,0.005861,0.003621,1
40895470,1020.999023,1270.407959,99.558998,59.0,67.0,56.0,44.0,1.0,1.0,0.0,...,0.664521,0.995267,0.922674,0.072947,0.273659,0.959971,0.007293,0.004733,0.002255,1
40895471,1021.242981,1270.141968,99.517998,81.0,91.0,80.0,23.0,1.0,1.0,0.0,...,0.667513,0.995349,0.778359,0.217299,0.336659,0.875990,0.007746,0.004651,0.000999,1
40895472,1020.203003,1270.517944,99.584999,58.0,66.0,55.0,74.0,1.0,1.0,0.0,...,0.669394,0.992829,0.861002,0.132204,0.297302,0.925863,0.011384,0.007171,0.001806,1


In [ ]:
print(data['label'].value_counts())

label
1    18358744
0    17485039
2     4900628
3      131227
4       19836
Name: count, dtype: int64


In [ ]:
#--------------------------------------------------------------------------------------------<Prepare data>------------------||
data = data.dropna(subset=['label'])

# x = data[['Column1','Column2','Column3','Column4','Column5','Column6','Column7','Column8']]
"""X = data[['z', 'eigenvalue_sum', 'omnivariance', 'eigenentropy',
       'anisotropy', 'planarity', 'linearity', 'PCA1', 'PCA2',
       'surface_variation', 'sphericity', 'verticality', 'nx', 'ny', 'nz']]""" #Toronto

X = data[['x', 'y', 'z', 'eigenvalue_sum', 'omnivariance', 'eigenentropy',
       'anisotropy', 'planarity', 'linearity', 'PCA1', 'PCA2',
       'surface_variation', 'sphericity', 'verticality']] #Toronto

y = data[['label']]

In [ ]:
#--------------------------------------------------------------------------------------------<Data split>--------------------||
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) #REVERTTHIS

X_train_values = X_train.values
y_train_values = y_train.values

In [ ]:
%%time
phi, mu, sigma, label_order = fit(X_train_values, y_train_values)
phi

CPU times: total: 18.6 s
Wall time: 3.98 s


array([0.42752647, 0.44888424, 0.11989716, 0.00320607, 0.00048606])

In [ ]:
import pickle

with open("F:\\Aditya\\Lidar Semantic Segmentation\\AHN3 Tiles\\tempDumps\\phi.pkl", "wb") as f:
    pickle.dump(X_train, f)

with open("F:\\Aditya\\Lidar Semantic Segmentation\\AHN3 Tiles\\tempDumps\\mu.pkl", "wb") as f:
    pickle.dump(X_test, f)

with open("F:\\Aditya\\Lidar Semantic Segmentation\\AHN3 Tiles\\tempDumps\\sigma.pkl", "wb") as f:
    pickle.dump(y_train, f)

In [ ]:
#--------------------------------------------------------------------------------------------<Multivariate gaussian pdf>-----||
import math
def multivariate_gaussian_pdf(x, mean, cov):
    d = mean.shape[0] #dimensionality of the input
    exponent = -0.5 * np.dot(np.dot((x - mean).T, np.linalg.inv(cov)), (x - mean)) # -(1/2) . Transpose(x−μ) . Inverse(Σ or Covariance Matx) . (x−μ)
    prefactor = 1 / np.sqrt(((2 * np.pi) ** d )*(np.linalg.det(cov))) # 1 / Sqrt( (2π)^d . |Σ| ) 
    return np.exp(exponent)*prefactor

In [ ]:
#--------------------------------------------------------------------------------------------<Positive semi definite chck>---||
# Covariance matrices must be positive semidefinite because variance (and covariance) can't be negative
def is_positive_semidefinite(matrix):
    eigenvalues, _ = np.linalg.eig(matrix)
    print(eigenvalues)
    return np.all(eigenvalues >= 0)

matrix = sigma[1] 
# print(matrix)
positive_semidefinite = is_positive_semidefinite(matrix)
if positive_semidefinite:
    print("The matrix is positive semidefinite.")
else:
    print("The matrix is not positive semidefinite.")

[9.21109997e+04 1.35488988e+05 4.07731636e+00 3.61051849e-02
 1.27410519e-02 1.16864769e-03 7.38823416e-04 6.94463042e-04
 1.09260781e-04 2.81456622e-05 1.83187499e-05 1.00000000e-05
 1.00000000e-05 1.00000000e-05]
The matrix is positive semidefinite.


In [ ]:
print(sigma)

[[[ 1.09033811e+05 -4.27422684e+03  1.06452888e+03  3.10403373e+00
    7.63163771e+00  5.10282851e+00 -1.10329197e+01 -6.69531551e+00
   -4.15617633e+00 -6.22939827e+00 -3.90079475e-01  1.31114139e+01
    1.10329195e+01  7.38640646e+00]
  [-4.27422684e+03  1.05506922e+05 -1.77831081e+03 -1.04535934e-01
    1.05267949e+00  5.04710014e-01 -1.95912559e+00 -1.75759209e+00
   -1.69487028e-01 -9.27825970e-01 -5.19249272e-01  2.65068009e+00
    1.95912552e+00  1.32717593e+00]
  [ 1.06452888e+03 -1.77831081e+03  6.74951876e+01  8.51212620e-02
    2.62173484e-01  1.66166723e-01 -3.71845865e-01 -2.87628778e-01
   -7.81208773e-02 -1.90056256e-01 -5.34155441e-02  4.62844391e-01
    3.71845858e-01  2.84672735e-01]
  [ 3.10403373e+00 -1.04535934e-01  8.51212620e-02  9.21938071e-03
    1.16922943e-02  8.79366581e-03 -8.02417326e-03 -1.26241994e-03
   -6.62872467e-03 -6.46198162e-03  2.69454322e-03  8.91116539e-03
    8.02417317e-03  2.78897483e-03]
  [ 7.63163771e+00  1.05267949e+00  2.62173484e-01  

In [ ]:
for i in range(sigma.shape[0]): 
    print(np.linalg.eigvals(sigma[i]))

[1.11925496e+05 1.02654275e+05 2.84825727e+01 1.53463392e-01
 1.02785769e-01 5.95798415e-02 1.52172832e-02 2.02572025e-03
 6.21406048e-04 2.66112998e-04 1.58923354e-04 9.99999995e-06
 1.00000000e-05 9.99999999e-06]
[9.21109997e+04 1.35488988e+05 4.07731636e+00 3.61051849e-02
 1.27410519e-02 1.16864769e-03 7.38823416e-04 6.94463042e-04
 1.09260781e-04 2.81456622e-05 1.83187499e-05 1.00000000e-05
 1.00000000e-05 1.00000000e-05]
[1.31552078e+05 4.99833785e+04 1.14962795e+01 1.41380552e-01
 4.75285272e-02 3.15379730e-02 4.25596375e-03 1.84097656e-03
 7.06465584e-04 8.63615445e-05 1.11096279e-04 1.00000000e-05
 1.00000000e-05 1.00000000e-05]
[8.35246277e+04 4.37568466e+04 4.63235414e+00 9.16158690e-02
 1.77315652e-02 2.86575310e-03 2.01680090e-03 1.42055200e-03
 6.58558813e-04 9.14034820e-05 3.18539758e-05 1.00000000e-05
 1.00000000e-05 1.00000000e-05]
[2.01145925e+05 4.45460286e+04 2.93281161e+00 1.62009405e-01
 3.81028337e-02 2.95652558e-02 3.37834817e-03 1.10122023e-03
 6.07830191e-04 1.

#### <span style="color:red">Epistemic uncertainty calculation</span>
Computes the epistemic uncertainty using this formula 1 − ∑ P( y=c | x )⋅ϕ(c)
- P( y=c | x ) is class probability from the Gaussian
- ϕ(c) is the prior (Φ, Prior probability of each class or Class prior probabilities (class frequencies))

In [ ]:
def give_epistemic_torch(X_df, mu, sigma, phi, device='cuda'):
    x_test = torch.tensor(X_df.values, dtype=torch.float32).to(device)
    mu_t = torch.tensor(mu, dtype=torch.float32).to(device)
    sigma_t = torch.tensor(sigma, dtype=torch.float32).to(device)
    phi_t = torch.tensor(phi, dtype=torch.float32).to(device)

    num_pts = x_test.shape[0]
    num_classes = sigma_t.shape[0]

    total_weighted_density = torch.zeros(num_pts, device=device)
    sum_of_densities = torch.zeros(num_pts, device=device)

    print(f"Calculating uncertainty for {num_pts} points on {device}...")

    for idx in range(num_classes):
        if phi_t[idx] > 0:
            dist = MultivariateNormal(loc=mu_t[idx], covariance_matrix=sigma_t[idx])
            p_x_given_c = torch.exp(dist.log_prob(x_test))
            
            weighted_p = p_x_given_c * phi_t[idx]
            sum_of_densities += weighted_p
            total_weighted_density += weighted_p * phi_t[idx]

    sum_of_densities = torch.clamp(sum_of_densities, min=1e-10)
    
    feature_density = total_weighted_density / sum_of_densities
    uncertainty = 1 - feature_density

    return uncertainty.cpu().numpy()


In [ ]:
%%time

#Compute epistemic
X_epistemic = give_epistemic_torch(X, mu, sigma, phi)

Calculating uncertainty for 40895474 points on cuda...
CPU times: total: 35.9 s
Wall time: 30.5 s


In [ ]:
X['epistemic'] = X_epistemic
data_new = pd.concat([X, y], axis=1)
data_new = data_new.dropna()
data_new.drop(columns=['eigenvalue_sum', 'PCA1', 'PCA2'], inplace=True)

In [ ]:
print(f"{fileName}: {data['label'].value_counts(normalize=True)}")
print(f"\n{fileName}_uncertainty: {data_new['label'].value_counts(normalize=True)}")

69EN2_08: label
1    0.448919
0    0.427554
2    0.119833
3    0.003209
4    0.000485
Name: proportion, dtype: float64

69EN2_08_uncertainty: label
1    0.448919
0    0.427554
2    0.119833
3    0.003209
4    0.000485
Name: proportion, dtype: float64


: 

In [ ]:
data_new.to_csv(rf"F:\Aditya\Lidar Semantic Segmentation\AHN3 Tiles\Uncertainty Added\{fileName}_uncertainty.csv", index=False)
data_new.columns

Index(['x', 'y', 'z', 'omnivariance', 'eigenentropy', 'anisotropy',
       'planarity', 'linearity', 'surface_variation', 'sphericity',
       'verticality', 'epistemic', 'label'],
      dtype='object')